# Raaga.AI — Pretrain + fine-tune SvaraTransformer on Colab

Runtime → Change runtime type → **T4 GPU**.

**Stage 1: pretrain** on large Hugging Face MIDI datasets. Each melody chunk is fitted to its closest raga.
| key | dataset | license | size |
|---|---|---|---|
| `lakh` | roszcz/lakh-lmd-full | CC-BY-4.0 (Lakh MIDI) | 174k songs, streamed |
| `classical-midi` | drengskapur/midi-classical-music | MIT | 4.8k files |
| `maestro` | roszcz/maestro-sustain-v2 | CC-BY-NC-SA-4.0 — non-commercial | 1.3k performances |

**Stage 2: fine-tune** on Carnatic data: Saraga 1.5 (bundled, CC-BY-NC-SA), local raga definitions, `kaumudi-ragas` (CC-BY-4.0), and optionally Dunya.

If you ship this commercially, drop `maestro` and Saraga. If you publish the model, credit the datasets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/raagaai'   # checkpoints + datasets persist here
!mkdir -p {DRIVE_DIR}/pretrain {DRIVE_DIR}/finetune {DRIVE_DIR}/training

In [ ]:
!test -d /content/raagaai || git clone -b feat/hf-training-hybrid-song https://github.com/Pariseashishbannu/raagaai.git /content/raagaai
%cd /content/raagaai
!pip install -q torch tqdm requests beautifulsoup4 numpy datasets huggingface_hub pretty_midi
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os
try:  # optional: HF_TOKEN / DUNYA_TOKEN in Colab Secrets (key icon, left sidebar)
    from google.colab import userdata
    for k in ('HF_TOKEN', 'DUNYA_TOKEN'):
        try: os.environ[k] = userdata.get(k) or ''
        except Exception: pass
except ImportError:
    pass

## 1. Build the pretraining set (large)
`HF_MAX_SONGS` = songs read per source. Each Lakh song gives about 15–40 sequences.
Lakh rows are big, so 20k songs means streaming a few GB (~20–40 min). That gives roughly 300k–800k sequences.
Colab RAM (12 GB) handles up to about 40k songs.

In [ ]:
HF_MAX_SONGS = 20000
PRE_SOURCES = 'lakh classical-midi maestro'

!python -m scripts.build_dataset --no-synthetic --no-augment \
    --hf {PRE_SOURCES} --hf-max-songs {HF_MAX_SONGS} \
    --output {DRIVE_DIR}/training/pretrain.jsonl
!wc -l {DRIVE_DIR}/training/pretrain.jsonl

## 2. Pretrain
Re-running this cell resumes from the Drive checkpoint after a disconnect.

In [ ]:
pre_ckpt = f'{DRIVE_DIR}/pretrain/svara_transformer.pt'
resume = f'--resume {pre_ckpt}' if os.path.exists(pre_ckpt) else ''

!python -m scripts.train_model \
    --data {DRIVE_DIR}/training/pretrain.jsonl \
    --output-dir {DRIVE_DIR}/pretrain \
    --epochs 20 --batch-size 256 --lr 5e-4 \
    --embed-dim 256 --num-heads 8 --num-layers 6 --max-seq-len 128 \
    --patience 4 --device cuda {resume}

## 3. Build the Carnatic fine-tuning set
Optional: upload `data/training/orange_audio.jsonl` (sequences pitch-tracked from your audio on your PC) to `MyDrive/raagaai/training/` and it gets merged in. If it's missing, it's skipped.

In [ ]:
!python -m scripts.build_dataset \
    --saraga-dir data/saraga/saraga1.5_carnatic \
    --hf kaumudi-ragas \
    --include-jsonl {DRIVE_DIR}/training/orange_audio.jsonl \
    --synthetic-per-raga 300 \
    --output {DRIVE_DIR}/training/finetune.jsonl
!wc -l {DRIVE_DIR}/training/finetune.jsonl

## 4. Fine-tune
Uses a lower learning rate and a fresh schedule. Model size comes from the pretrained checkpoint, and new raga tokens get embedding rows added automatically.

In [ ]:
ft_ckpt = f'{DRIVE_DIR}/finetune/svara_transformer.pt'
start = f'--resume {ft_ckpt}' if os.path.exists(ft_ckpt) else f'--finetune {pre_ckpt}'

!python -m scripts.train_model \
    --data {DRIVE_DIR}/training/finetune.jsonl \
    --output-dir {DRIVE_DIR}/finetune \
    --epochs 30 --batch-size 128 --lr 1e-4 --min-lr 1e-5 \
    --patience 6 --device cuda {start}

## 5. Check + download
Put both files in `data/models/` locally.

In [ ]:
from src.ml.model import load_model

model, config = load_model(ft_ckpt, device='cuda')
print(config)
print(f'{model.count_parameters():,} parameters')

from google.colab import files
files.download(ft_ckpt)
files.download(f'{DRIVE_DIR}/finetune/tokenizer_vocab.json')